# 18장 자기소개서 도우미 챗봇

이 장은 PDF 교재의 LangChain, PromptTemplate, Gradio 챗봇 실습 흐름을 바탕으로 자기소개서 문장을 분석하고 개선하는 LLM 챗봇을 만드는 방법을 정리합니다.

자기소개서 도우미 챗봇은 사용자가 작성한 문장을 입력받아 문장의 강점, 보완점, 개선 예시, 면접 질문 가능성을 안내하는 응용 실습입니다. 이 장은 앞에서 배운 LLM 호출, 프롬프트 설계, 출력 형식 제어, Gradio UI 구성을 하나의 실전형 예제로 연결합니다.

## 이 장에서 다루는 내용

1. 자기소개서 도우미 챗봇의 목표
2. 자기소개서 피드백 기준
3. 설치와 기본 import
4. 입력 데이터 구조 설계
5. 기본 피드백 프롬프트 만들기
6. LangChain으로 피드백 체인 구성
7. 항목별 피드백 함수 만들기
8. 개선 문장 생성 함수 만들기
9. 면접 질문 생성 함수 만들기
10. 종합 분석 챗봇 만들기
11. Gradio UI 구성
12. 개인정보 보호와 사용 시 주의점
13. 오류 해결과 확장 방향

이 장은 `LLM/llm.ipynb`의 LangChain 체인과 Gradio 챗봇 실습을 문서 첨삭 서비스 형태로 확장한 내용입니다.


## 18.1 자기소개서 도우미 챗봇의 목표

자기소개서 도우미 챗봇의 목표는 단순히 문장을 예쁘게 바꾸는 것이 아닙니다. 지원자의 경험이 더 명확하게 전달되도록 돕고, 문항 의도에 맞게 구조를 개선하는 것입니다.

챗봇이 제공할 수 있는 기능은 다음과 같습니다.

- 문항 의도 분석
- 자기소개서 문장 피드백
- 경험의 구체성 점검
- 직무 연관성 점검
- 문장 흐름과 논리 구조 개선
- 더 나은 수정 예시 생성
- 예상 면접 질문 생성

이 실습에서는 LLM이 지원자의 내용을 대신 만들어내는 방식보다, 사용자가 작성한 내용을 바탕으로 더 명확하고 설득력 있게 다듬는 방향을 기준으로 합니다.


## 18.2 자기소개서 피드백 기준

자기소개서를 평가할 때는 다음 기준을 사용할 수 있습니다.

| 기준 | 설명 |
|---|---|
| 문항 적합성 | 질문에서 요구하는 내용을 실제로 답하고 있는가 |
| 구체성 | 경험, 행동, 결과가 구체적으로 드러나는가 |
| 직무 연관성 | 지원 직무와 연결되는 역량이 보이는가 |
| 논리성 | 문제, 행동, 결과, 배움의 흐름이 자연스러운가 |
| 진정성 | 과장된 표현보다 실제 경험 중심으로 작성되었는가 |
| 가독성 | 문장이 너무 길거나 모호하지 않은가 |
| 차별성 | 지원자만의 역할과 기여가 드러나는가 |

프롬프트를 만들 때 이 기준을 명시하면 LLM이 더 일관된 피드백을 제공할 수 있습니다.


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 설치 준비
# 이 셀은 자기소개서 도우미 챗봇 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# langchain은 프롬프트, 모델, 파서를 연결하는 체인 구성을 위해 사용합니다.
!pip install langchain

# langchain-ollama는 Ollama 로컬 LLM을 LangChain 방식으로 호출할 때 사용합니다.
!pip install langchain-ollama

# gradio는 자기소개서 도우미 챗봇 웹 UI를 만들 때 사용합니다.
!pip install gradio

# pandas는 예시 데이터나 피드백 결과를 표 형태로 관리할 때 사용할 수 있습니다.
!pip install pandas


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 기본 import
# 이 셀은 실습에 필요한 Python 모듈과 LangChain 구성 요소를 불러옵니다.

# ChatPromptTemplate은 채팅 모델에 전달할 프롬프트 템플릿을 만듭니다.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser는 LLM 응답 객체를 문자열로 변환합니다.
from langchain_core.output_parsers import StrOutputParser

# ChatOllama는 Ollama에서 실행 중인 로컬 LLM을 LangChain 인터페이스로 호출합니다.
from langchain_ollama import ChatOllama

# gradio는 웹 기반 입력과 출력 화면을 만들 때 사용합니다.
import gradio as gr

# pandas는 결과를 표 형태로 다룰 때 사용합니다.
import pandas as pd


## 18.3 입력 데이터 구조 설계

자기소개서 피드백을 하려면 사용자의 글만 받는 것보다 문항, 지원 직무, 회사명 같은 맥락을 함께 받는 것이 좋습니다.

입력 항목 예시는 다음과 같습니다.

| 입력 항목 | 설명 |
|---|---|
| company | 지원 회사명 |
| role | 지원 직무 |
| question | 자기소개서 문항 |
| answer | 사용자가 작성한 자기소개서 답변 |
| tone | 원하는 문체 또는 피드백 강도 |

문항과 직무 정보를 함께 넣으면 LLM이 자기소개서를 더 정확하게 평가할 수 있습니다.


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 예시 입력 데이터
# 이 셀은 자기소개서 피드백 함수에 넣을 예시 데이터를 준비합니다.

# sample_company 변수에는 지원 회사명을 저장합니다.
sample_company = "예시테크"

# sample_role 변수에는 지원 직무명을 저장합니다.
sample_role = "데이터 분석가"

# sample_question 변수에는 자기소개서 문항을 저장합니다.
sample_question = "지원 직무와 관련된 경험과 본인의 강점을 설명해 주세요."

# sample_answer 변수에는 사용자가 작성한 자기소개서 답변 예시를 저장합니다.
sample_answer = "저는 데이터를 다루는 것이 재미있어서 관련 프로젝트를 했습니다. 팀 프로젝트에서 데이터를 분석했고 좋은 결과를 얻었습니다. 앞으로 회사에서도 열심히 하겠습니다."

# sample_tone 변수에는 원하는 피드백 톤을 저장합니다.
sample_tone = "친절하지만 구체적으로"

# 예시 입력값을 확인하기 위해 출력합니다.
print(sample_company, sample_role, sample_question, sample_answer, sample_tone)


## 18.4 기본 피드백 프롬프트 만들기

자기소개서 피드백 프롬프트에는 역할, 평가 기준, 출력 형식을 명확히 넣는 것이 좋습니다.

좋은 프롬프트 구성은 다음과 같습니다.

```text
역할: 너는 자기소개서 코치다.
맥락: 회사, 직무, 문항, 작성 답변을 제공한다.
기준: 문항 적합성, 구체성, 직무 연관성, 논리성, 가독성.
출력: 강점, 보완점, 개선 방향, 수정 예시.
주의: 없는 경험을 새로 만들지 않는다.
```

특히 자기소개서에서는 사실이 아닌 내용을 만들어내면 안 되므로, 사용자가 제공한 내용 안에서 개선하도록 지시해야 합니다.


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - LLM과 피드백 프롬프트
# 이 셀은 자기소개서 피드백에 사용할 LLM과 프롬프트를 구성합니다.

# llm 변수에는 사용할 Ollama 로컬 모델을 지정합니다.
llm = ChatOllama(model="llama3.1")

# feedback_prompt는 자기소개서 피드백을 생성하기 위한 채팅 프롬프트입니다.
feedback_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 모델의 역할과 중요한 제한 사항을 지정합니다.
    ("system", "너는 한국어 자기소개서 코치야. 사용자가 제공한 사실을 바탕으로만 피드백하고, 없는 경험을 새로 지어내지 마."),
    # human 메시지는 회사, 직무, 문항, 답변, 톤을 변수로 받아 피드백을 요청합니다.
    ("human", "회사: {company}\n직무: {role}\n문항: {question}\n작성 답변:\n{answer}\n\n피드백 톤: {tone}\n\n다음 형식으로 답변해줘.\n1. 전체 평가\n2. 강점\n3. 보완점\n4. 구체화하면 좋은 부분\n5. 수정 예시\n6. 추가로 작성하면 좋은 소재")
])

# feedback_chain은 프롬프트, LLM, 문자열 파서를 연결한 체인입니다.
feedback_chain = feedback_prompt | llm | StrOutputParser()


## 18.5 기본 피드백 함수 만들기

피드백 함수는 Gradio UI와 연결하기 쉽게 입력값을 받아 문자열 답변을 반환하도록 만듭니다.

함수 내부에서는 빈 입력을 확인하고, 입력값을 LangChain 체인에 전달합니다.


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 기본 피드백 함수
# 이 셀은 자기소개서 답변에 대한 피드백을 생성하는 함수를 정의합니다.

# generate_feedback 함수는 회사, 직무, 문항, 답변, 톤을 받아 피드백을 생성합니다.
def generate_feedback(company: str, role: str, question: str, answer: str, tone: str) -> str:
    # 자기소개서 답변이 비어 있으면 안내 문장을 반환합니다.
    if not answer or not answer.strip():
        # 입력 답변이 없을 때의 안내 메시지입니다.
        return "자기소개서 답변을 입력해 주세요."

    # 회사명이 비어 있으면 기본값을 사용합니다.
    company = company or "지원 회사"

    # 직무명이 비어 있으면 기본값을 사용합니다.
    role = role or "지원 직무"

    # 문항이 비어 있으면 기본 문항을 사용합니다.
    question = question or "자기소개서 문항이 제공되지 않았습니다."

    # 톤이 비어 있으면 기본 피드백 톤을 사용합니다.
    tone = tone or "친절하고 구체적으로"

    # feedback_chain에 입력값을 전달해 LLM 피드백을 생성합니다.
    feedback = feedback_chain.invoke({
        # company는 지원 회사명입니다.
        "company": company,
        # role은 지원 직무명입니다.
        "role": role,
        # question은 자기소개서 문항입니다.
        "question": question,
        # answer는 사용자가 작성한 자기소개서 답변입니다.
        "answer": answer,
        # tone은 원하는 피드백 스타일입니다.
        "tone": tone,
    })

    # 생성된 피드백 문자열을 반환합니다.
    return feedback


# 예시 입력값으로 피드백 함수를 호출합니다.
sample_feedback = generate_feedback(sample_company, sample_role, sample_question, sample_answer, sample_tone)

# 생성된 피드백을 출력합니다.
print(sample_feedback)


## 18.6 항목별 평가 프롬프트 만들기

종합 피드백도 유용하지만, 항목별 점검표 형태로 평가하면 사용자가 부족한 부분을 더 쉽게 파악할 수 있습니다.

항목별 평가 예시는 다음과 같습니다.

- 문항 적합성
- 경험의 구체성
- 직무 연관성
- 성과와 결과
- 문장 가독성
- 차별성

점수는 절대적인 합격 기준이 아니라 개선 방향을 찾기 위한 참고 지표로 사용해야 합니다.


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 항목별 평가 체인
# 이 셀은 자기소개서를 항목별로 평가하는 프롬프트와 체인을 만듭니다.

# score_prompt는 평가 항목별 점수와 이유를 생성하기 위한 프롬프트입니다.
score_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 평가자의 역할을 지정합니다.
    ("system", "너는 자기소개서 평가표를 작성하는 한국어 평가자야. 점수는 개선 참고용으로만 제시해."),
    # human 메시지는 평가 대상 정보를 전달하고 표 형식 출력을 요청합니다.
    ("human", "직무: {role}\n문항: {question}\n답변:\n{answer}\n\n문항 적합성, 구체성, 직무 연관성, 성과 표현, 가독성, 차별성을 각각 1~5점으로 평가하고 이유를 표 형태로 작성해줘.")
])

# score_chain은 항목별 평가 프롬프트와 LLM, 출력 파서를 연결합니다.
score_chain = score_prompt | llm | StrOutputParser()


# evaluate_resume_answer 함수는 자기소개서 답변을 항목별로 평가합니다.
def evaluate_resume_answer(role: str, question: str, answer: str) -> str:
    # 답변이 비어 있으면 안내 문장을 반환합니다.
    if not answer or not answer.strip():
        # 평가할 답변이 없다는 안내입니다.
        return "평가할 자기소개서 답변을 입력해 주세요."

    # 직무가 비어 있으면 기본값을 사용합니다.
    role = role or "지원 직무"

    # 문항이 비어 있으면 기본값을 사용합니다.
    question = question or "자기소개서 문항이 제공되지 않았습니다."

    # score_chain을 실행해 항목별 평가 결과를 생성합니다.
    evaluation = score_chain.invoke({"role": role, "question": question, "answer": answer})

    # 평가 결과 문자열을 반환합니다.
    return evaluation


## 18.7 개선 문장 생성 함수 만들기

자기소개서 도우미는 단순 지적보다 개선 방향을 제시해야 유용합니다. 다만 사용자가 쓰지 않은 경험을 새로 만들어내면 안 됩니다.

개선 문장 생성 프롬프트에는 다음 제한을 넣습니다.

- 원문에 없는 경력, 성과, 수치를 새로 만들지 않기
- 모호한 표현을 구체화할 질문을 함께 제시하기
- 원문의 의미를 유지하되 구조와 표현을 다듬기
- 너무 과장된 표현을 피하기


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 개선 문장 생성 체인
# 이 셀은 자기소개서 답변의 수정 예시를 생성하는 체인을 만듭니다.

# rewrite_prompt는 원문을 더 명확한 자기소개서 문장으로 다듬기 위한 프롬프트입니다.
rewrite_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 수정 원칙을 지정합니다.
    ("system", "너는 자기소개서 문장 개선 전문가야. 원문에 없는 사실, 수치, 경험을 새로 만들지 말고 표현과 구조만 개선해."),
    # human 메시지는 직무, 문항, 원문을 전달하고 수정본 생성을 요청합니다.
    ("human", "직무: {role}\n문항: {question}\n원문:\n{answer}\n\n다음 형식으로 작성해줘.\n1. 다듬은 자기소개서 문장\n2. 왜 이렇게 수정했는지\n3. 추가로 사용자가 채우면 좋은 구체 정보")
])

# rewrite_chain은 수정 프롬프트와 LLM, 출력 파서를 연결합니다.
rewrite_chain = rewrite_prompt | llm | StrOutputParser()


# rewrite_resume_answer 함수는 자기소개서 답변의 개선 예시를 생성합니다.
def rewrite_resume_answer(role: str, question: str, answer: str) -> str:
    # 원문 답변이 비어 있으면 안내 문장을 반환합니다.
    if not answer or not answer.strip():
        # 수정할 답변이 없다는 안내입니다.
        return "수정할 자기소개서 답변을 입력해 주세요."

    # 직무가 비어 있으면 기본값을 사용합니다.
    role = role or "지원 직무"

    # 문항이 비어 있으면 기본값을 사용합니다.
    question = question or "자기소개서 문항이 제공되지 않았습니다."

    # rewrite_chain을 실행해 개선 문장을 생성합니다.
    rewritten = rewrite_chain.invoke({"role": role, "question": question, "answer": answer})

    # 개선 결과 문자열을 반환합니다.
    return rewritten


## 18.8 예상 면접 질문 생성

자기소개서가 완성되면 면접관이 어떤 질문을 할지 예상해 보는 것이 좋습니다.

예상 면접 질문은 다음 관점에서 만들 수 있습니다.

- 경험의 구체적인 상황 확인
- 본인의 실제 역할 확인
- 성과와 수치의 근거 확인
- 실패 또는 갈등 상황 확인
- 직무 역량과 연결되는 부분 확인
- 입사 후 어떻게 활용할지 확인


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 예상 면접 질문 생성 체인
# 이 셀은 자기소개서 답변을 바탕으로 예상 면접 질문을 생성합니다.

# interview_prompt는 예상 면접 질문을 생성하기 위한 프롬프트입니다.
interview_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 면접관 역할을 지정합니다.
    ("system", "너는 지원자의 자기소개서를 읽고 예상 면접 질문을 만드는 한국어 면접관이야."),
    # human 메시지는 직무, 문항, 답변을 전달하고 질문 생성을 요청합니다.
    ("human", "직무: {role}\n문항: {question}\n자기소개서 답변:\n{answer}\n\n예상 면접 질문 8개와 각 질문의 의도를 함께 작성해줘.")
])

# interview_chain은 면접 질문 프롬프트와 LLM, 출력 파서를 연결합니다.
interview_chain = interview_prompt | llm | StrOutputParser()


# generate_interview_questions 함수는 예상 면접 질문을 생성합니다.
def generate_interview_questions(role: str, question: str, answer: str) -> str:
    # 자기소개서 답변이 비어 있으면 안내 문장을 반환합니다.
    if not answer or not answer.strip():
        # 질문을 만들 답변이 없다는 안내입니다.
        return "예상 면접 질문을 만들 자기소개서 답변을 입력해 주세요."

    # 직무가 비어 있으면 기본값을 사용합니다.
    role = role or "지원 직무"

    # 문항이 비어 있으면 기본값을 사용합니다.
    question = question or "자기소개서 문항이 제공되지 않았습니다."

    # interview_chain을 실행해 예상 면접 질문을 생성합니다.
    interview_questions = interview_chain.invoke({"role": role, "question": question, "answer": answer})

    # 생성된 예상 면접 질문을 반환합니다.
    return interview_questions


## 18.9 종합 분석 함수 만들기

Gradio UI에서는 버튼 하나로 종합 피드백, 항목별 평가, 개선 문장, 예상 면접 질문을 모두 보여줄 수 있습니다.

다만 실제 서비스에서는 한 번에 너무 많은 LLM 호출을 하면 시간이 오래 걸리고 비용이 증가할 수 있습니다. 실습에서는 기능을 이해하기 위해 종합 함수를 구성합니다.


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - 종합 분석 함수
# 이 셀은 여러 분석 기능을 한 번에 실행하는 함수를 정의합니다.

# analyze_resume_all 함수는 자기소개서 답변을 종합적으로 분석합니다.
def analyze_resume_all(company: str, role: str, question: str, answer: str, tone: str) -> tuple[str, str, str, str]:
    # 답변이 비어 있으면 모든 출력에 안내 문장을 반환합니다.
    if not answer or not answer.strip():
        # 빈 입력에 대한 안내 메시지입니다.
        message = "자기소개서 답변을 입력해 주세요."

        # 네 개 출력 영역에 같은 안내 메시지를 반환합니다.
        return message, message, message, message

    # 종합 피드백을 생성합니다.
    feedback = generate_feedback(company, role, question, answer, tone)

    # 항목별 평가를 생성합니다.
    evaluation = evaluate_resume_answer(role, question, answer)

    # 개선 문장 예시를 생성합니다.
    rewritten = rewrite_resume_answer(role, question, answer)

    # 예상 면접 질문을 생성합니다.
    interview = generate_interview_questions(role, question, answer)

    # 네 가지 결과를 순서대로 반환합니다.
    return feedback, evaluation, rewritten, interview


## 18.10 Gradio 자기소개서 도우미 UI

Gradio를 사용하면 사용자가 회사, 직무, 문항, 답변을 입력하고 결과를 탭별로 확인하는 UI를 만들 수 있습니다.

UI 구성은 다음과 같습니다.

```text
회사명 / 직무 / 문항 / 자기소개서 답변 / 피드백 톤 입력
  -> 종합 분석 실행
  -> 종합 피드백
  -> 항목별 평가
  -> 개선 문장
  -> 예상 면접 질문
```


In [ ]:
# 교재 위치: 18장 자기소개서 도우미 챗봇 - Gradio UI
# 이 셀은 자기소개서 도우미 챗봇의 웹 UI를 구성합니다.

# Blocks는 여러 입력과 출력 컴포넌트를 자유롭게 배치할 수 있는 Gradio 컨테이너입니다.
with gr.Blocks() as resume_demo:
    # Markdown은 화면 제목을 표시합니다.
    gr.Markdown("# 자기소개서 도우미 챗봇")

    # Row는 여러 입력 컴포넌트를 한 줄에 배치할 때 사용합니다.
    with gr.Row():
        # 회사명 입력 Textbox를 만듭니다.
        company_input = gr.Textbox(label="지원 회사", value=sample_company)

        # 직무 입력 Textbox를 만듭니다.
        role_input = gr.Textbox(label="지원 직무", value=sample_role)

    # 자기소개서 문항 입력 Textbox를 만듭니다.
    question_input = gr.Textbox(label="자기소개서 문항", lines=2, value=sample_question)

    # 자기소개서 답변 입력 Textbox를 만듭니다.
    answer_input = gr.Textbox(label="작성한 자기소개서 답변", lines=10, value=sample_answer)

    # 피드백 톤 입력 Textbox를 만듭니다.
    tone_input = gr.Textbox(label="피드백 톤", value=sample_tone)

    # 분석 실행 버튼을 만듭니다.
    analyze_button = gr.Button("자기소개서 분석하기")

    # Tabs는 여러 출력 결과를 탭으로 나누어 보여줍니다.
    with gr.Tabs():
        # 종합 피드백 탭을 만듭니다.
        with gr.Tab("종합 피드백"):
            # 종합 피드백 출력 Textbox를 만듭니다.
            feedback_output = gr.Textbox(label="종합 피드백", lines=18)

        # 항목별 평가 탭을 만듭니다.
        with gr.Tab("항목별 평가"):
            # 항목별 평가 출력 Textbox를 만듭니다.
            evaluation_output = gr.Textbox(label="항목별 평가", lines=18)

        # 개선 문장 탭을 만듭니다.
        with gr.Tab("개선 문장"):
            # 개선 문장 출력 Textbox를 만듭니다.
            rewrite_output = gr.Textbox(label="개선 문장", lines=18)

        # 예상 면접 질문 탭을 만듭니다.
        with gr.Tab("예상 면접 질문"):
            # 예상 면접 질문 출력 Textbox를 만듭니다.
            interview_output = gr.Textbox(label="예상 면접 질문", lines=18)

    # 버튼 클릭 시 analyze_resume_all 함수를 실행합니다.
    analyze_button.click(
        # fn에는 실행할 종합 분석 함수를 지정합니다.
        fn=analyze_resume_all,
        # inputs에는 함수에 전달할 입력 컴포넌트들을 순서대로 지정합니다.
        inputs=[company_input, role_input, question_input, answer_input, tone_input],
        # outputs에는 함수 반환값을 받을 출력 컴포넌트들을 순서대로 지정합니다.
        outputs=[feedback_output, evaluation_output, rewrite_output, interview_output],
    )

# launch를 실행하면 로컬 웹 브라우저에서 자기소개서 도우미 챗봇을 확인할 수 있습니다.
# resume_demo.launch()


## 18.11 개인정보 보호와 사용 시 주의점

자기소개서에는 개인정보와 민감한 경험이 포함될 수 있습니다. 따라서 실습과 서비스에서 다음을 주의해야 합니다.

- 주민등록번호, 전화번호, 주소 같은 직접 식별 정보는 입력하지 않습니다.
- 회사 내부 정보나 비밀 프로젝트 내용은 입력하지 않습니다.
- LLM이 제안한 수정 문장을 그대로 제출하기보다 본인의 경험과 표현으로 다시 확인합니다.
- 없는 경험, 수치, 성과를 생성하지 않도록 프롬프트에 제한을 둡니다.
- 결과물은 참고용이며 최종 책임은 작성자에게 있습니다.
- 저장 기능을 만들 경우 원문과 분석 결과의 보관 기간을 정해야 합니다.

교재 실습에서는 로컬 모델과 예시 데이터를 사용해 구조를 익히고, 실제 자기소개서에는 민감 정보를 줄여 입력하는 습관을 갖는 것이 좋습니다.


## 18.12 확장 아이디어

자기소개서 도우미 챗봇은 다음 방향으로 확장할 수 있습니다.

- 여러 문항을 한 번에 분석하기
- STAR 기법에 맞춰 구조화하기
- 직무별 평가 기준을 다르게 적용하기
- 회사 인재상과 자기소개서 답변 연결하기
- PDF나 Word 파일 업로드 후 자동 피드백하기
- 피드백 전후 문장 비교표 만들기
- LangSmith로 피드백 품질 추적하기
- LangGraph로 피드백, 수정, 면접 질문 생성 흐름을 노드화하기
- RAG를 사용해 회사 채용 공고나 직무 설명을 근거로 피드백하기


## 18.13 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| Ollama 호출 실패 | Ollama 서버 미실행 | Ollama 실행 후 모델 이름 확인 |
| 답변이 너무 일반적임 | 직무와 문항 정보 부족 | 회사, 직무, 문항, 작성 답변을 더 구체적으로 입력 |
| 없는 경험을 만들어냄 | 프롬프트 제한 부족 | 제공한 사실만 사용하라는 지시 강화 |
| 출력 형식이 흔들림 | 형식 지시가 약함 | 번호 목록이나 표 형식을 명확히 요구 |
| 실행 시간이 김 | 여러 LLM 체인을 한 번에 실행 | 탭별 버튼 분리 또는 모델 경량화 |
| Gradio 출력이 비어 있음 | 함수 반환값 개수와 output 개수 불일치 | 반환 tuple과 outputs 목록 확인 |


## 18.14 정리

이 장에서는 LangChain과 Gradio를 사용해 자기소개서 도우미 챗봇을 만드는 방법을 정리했습니다.

핵심 정리는 다음과 같습니다.

- 자기소개서 도우미 챗봇은 LLM의 문서 피드백 능력을 활용하는 실전 응용입니다.
- 회사, 직무, 문항, 작성 답변을 함께 입력하면 더 정확한 피드백을 받을 수 있습니다.
- 프롬프트에는 평가 기준과 출력 형식을 명확히 넣어야 합니다.
- 사용자가 제공하지 않은 경험이나 수치를 새로 만들지 않도록 제한해야 합니다.
- 종합 피드백, 항목별 평가, 개선 문장, 예상 면접 질문을 각각 체인으로 구성할 수 있습니다.
- Gradio를 사용하면 자기소개서 분석 UI를 빠르게 만들 수 있습니다.
- 개인정보와 민감 정보를 입력하지 않도록 주의해야 합니다.

다음 장에서는 파인튜닝, 지식증류, RAG 비교를 다룹니다. LLM을 특정 목적에 맞게 활용하는 여러 전략의 차이를 정리합니다.
